# Section 1: Environment Setup & Package Imports

In this section, we install required third-party packages (such as `nonconformist` for conformal prediction) and import essential libraries:
- **Data Manipulation & I/O**: `pandas`, `numpy`, `os`, `pickle`
- **Visualization**: `matplotlib`, `seaborn`, `plotly`
- **Machine Learning Models & Metrics**: `scikit-learn` (Lasso, DecisionTree, RandomForest, metrics), `xgboost` (XGBRegressor)
- **Conformal Prediction**: `nonconformist` (ICP, RegressorNc, RegressorNormalizer)


### Step 1.1: Install Dependencies
Install the `nonconformist`, `xgboost`, and `plotly` packages.

In [ ]:
%pip install nonconformist xgboost plotly


### Step 1.2: Import Core Libraries & Configure Global Settings
Import all analysis libraries, configure warning suppression, pandas display options, and random seed.

In [ ]:
# System & Core Data Tools
import os
import numpy as np
import pandas as pd
from tqdm import tqdm
import pickle
import warnings
warnings.filterwarnings('ignore')

# Visualization Tools
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
try:
    import plotly.express as px
except ImportError:
    px = None

# Scikit-Learn Model & Evaluation Tools
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.feature_selection import mutual_info_regression, SelectFromModel
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score, RepeatedKFold, TimeSeriesSplit
from sklearn.linear_model import Lasso, LassoCV, LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor, export_text

# XGBoost
import xgboost as xgb
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error as MSE

# Conformal Prediction Tools
from nonconformist.icp import IcpClassifier, IcpRegressor
from nonconformist.nc import (
    ClassifierNc, MarginErrFunc, ClassifierAdapter, RegressorNc, 
    AbsErrorErrFunc, NcFactory, RegressorNormalizer
)
from nonconformist.cp import IcpRegressor

# Statsmodels
import statsmodels.api as sm
import statsmodels.formula.api as smf

# Configure display and warning settings
pd.set_option('display.max_columns', None)
%matplotlib inline
np.random.seed(1)


# Section 2: Data Ingestion & Dataset Preparation

In this section, we load the 2026 dataset Excel files and separate them into training subsets and specific player target slices (Corbin Carroll dataset slices `CB` and `CBB`).

### Step 2.1: Load MLB Main Project Data (2026 Version)
Load `MLB project 2026 version.xlsx` and separate general player rows (`data1`) from Corbin Carroll specific target rows (`CBB`).

In [ ]:
data = pd.read_excel('MLB project 2026 version.xlsx')
data11 = data.drop(['id'], axis=1, errors='ignore')
data1 = data11[:42204]
CBB = data11[42204:]
data1.head()


### Step 2.2: Load MLB All-Time Data (2026 Version)
Load `MLB all time data new 2026 version.xlsx` and separate all-time historical data (`alltime1`) from the target slice (`CB`).

In [ ]:
alltime = pd.read_excel('MLB all time data new 2026 version.xlsx')
alltime1 = alltime[:7082]
CB = alltime[7082:]
alltime1.head()


### Step 2.3: Load Future Career Projections Template (2026 Version)
Load `future 2026 version.xlsx` which contains the feature template for forward-looking career projections.

In [ ]:
future = pd.read_excel('future 2026 version.xlsx')
future


# Section 3: Baseline Feature Selection & Lasso Regression Modeling

Here we build linear models with L1 regularization (Lasso) to predict future player performance (`WAR` and career `year`) and inspect feature importance coefficients.

### Step 3.1: Train/Test Split for WAR Prediction
Sample 80% training data and 20% testing data from `data1`, dropping non-feature identifier and test columns.

In [ ]:
n_train = data1.sample(frac=0.8, replace=False, axis=0, random_state=1)
n_test = data1.sample(frac=0.2, replace=False, axis=0, random_state=1)

drop_cols_proj = ['Season', 'playerid', 'Name', 'Team', 'WAR', 'Career', 'total', 'remain', 'CDF', 
                  'WAR-1(test)', 'AB-1(test)', 'H-1(test)', 'HR-1(test)', 'RBI-1(test)', 'SB-1(test)', 'AVG-1(test)']

x_train = n_train.drop(drop_cols_proj, axis=1, errors='ignore').select_dtypes(include=[np.number]).fillna(0)
y_train = n_train['WAR'].fillna(0)

x_test = n_test.drop(drop_cols_proj, axis=1, errors='ignore').select_dtypes(include=[np.number]).fillna(0)
y_test = n_test['WAR'].fillna(0)


### Step 3.2: Fit Initial Baseline Lasso Model
Fit a Lasso regression model with default regularization hyperparameter $\alpha = 1$.

In [ ]:
model = Lasso(alpha=1)
model.fit(x_train, y_train)

# Evaluate initial model performance
print('R squared training set:', round(model.score(x_train, y_train) * 100, 2))
print('R squared test set:', round(model.score(x_test, y_test) * 100, 2))

pred_train = model.predict(x_train)
mse_train = mean_squared_error(y_train, pred_train)
print('MSE training set:', round(mse_train, 2))

pred = model.predict(x_test)
mse_test = mean_squared_error(y_test, pred)
print('MSE test set:', round(mse_test, 2))


### Step 3.3: Hyperparameter Exploration (Lasso Regularization Path)
Fit Lasso models across a logarithmic scale of alpha values to observe feature selection shrinkage dynamics.

In [ ]:
alphas = np.linspace(0.01, 10, 100)
lasso = Lasso(max_iter=10000)
coefs = []

for a in alphas:
    lasso.set_params(alpha=a)
    lasso.fit(x_train, y_train)
    coefs.append(lasso.coef_)

ax = plt.gca()
ax.plot(alphas, coefs)
ax.set_xscale('log')
plt.axis('tight')
plt.xlabel('alpha')
plt.ylabel('Standard Coefficients')
plt.title('Lasso coefficients as a function of alpha')
plt.show()


### Step 3.4: 5-Fold Cross-Validated Lasso (LassoCV)
Determine the optimal regularization parameter $\alpha$ using 5-fold cross-validation.

In [ ]:
model = LassoCV(cv=5, random_state=0, max_iter=10000)
model.fit(x_train, y_train)

print("Best Alpha found by CV:", model.alpha_)

# Fit best model
lasso_best = Lasso(alpha=model.alpha_)
lasso_best.fit(x_train, y_train)

print('R squared training set:', round(lasso_best.score(x_train, y_train) * 100, 2))
print('R squared test set:', round(lasso_best.score(x_test, y_test) * 100, 2))
print('Test MSE:', mean_squared_error(y_test, lasso_best.predict(x_test)))

# Display non-zero feature coefficients
print("\nTop Feature Coefficients:")
for coef, feat in zip(lasso_best.coef_, x_train.columns):
    if abs(coef) > 1e-4:
        print(f"  {feat}: {coef:.4f}")


### Step 3.5: Plot LassoCV Mean Squared Error Path Across Folds
Visualize the mean squared error for each fold and average across folds versus penalty $\alpha$.

In [ ]:
plt.semilogx(model.alphas_, model.mse_path_, ":")
plt.plot(model.alphas_, model.mse_path_.mean(axis=-1), "k", label="Average across the folds", linewidth=2)
plt.axvline(model.alpha_, linestyle="--", color="k", label="alpha: CV estimate")

plt.legend()
plt.xlabel('alphas')
plt.ylabel('Mean Square Error')
plt.title('Mean Square Error on Each Fold')
plt.axis('tight')
plt.show()


### Step 3.6: Year-Based Lasso Modeling & Cross-Validation
Prepare year-based training splits on `alltime1` and fit LassoCV to predict player `year` trajectory.

In [ ]:
# Prepare year dataset features
n_train_year = alltime1.sample(frac=0.8, replace=False, axis=0, random_state=1)
n_test_year = alltime1.sample(frac=0.2, replace=False, axis=0, random_state=1)

drop_cols_alltime = ['playerid', 'Name', 'Team', 'xwOBA', 'year', 'WAR', 'wRC', 'count', 'mlbamid', 
                     'Grade WAR', 'Grade AB', 'Grade H', 'Grade HR', 'Grade RBI', 'Grade SB', 'active']

x_train_year = n_train_year.drop(drop_cols_alltime, axis=1, errors='ignore').select_dtypes(include=[np.number]).fillna(0)
y_train_year = n_train_year['year'].fillna(0)
x_test_year = n_test_year.drop(drop_cols_alltime, axis=1, errors='ignore').select_dtypes(include=[np.number]).fillna(0)
y_test_year = n_test_year['year'].fillna(0)

model_year = LassoCV(cv=5, random_state=0, max_iter=10000)
model_year.fit(x_train_year, y_train_year)

lasso_best_year = Lasso(alpha=model_year.alpha_)
lasso_best_year.fit(x_train_year, y_train_year)

print('Best Alpha:', model_year.alpha_)
print('R squared training set:', round(lasso_best_year.score(x_train_year, y_train_year) * 100, 2))
print('R squared test set:', round(lasso_best_year.score(x_test_year, y_test_year) * 100, 2))
print('Test MSE:', mean_squared_error(y_test_year, lasso_best_year.predict(x_test_year)))


# Section 4: Non-linear Machine Learning with XGBoost

In this section, we replace linear modeling with non-linear Gradient Boosted Trees (`XGBRegressor`) to capture non-linear relationships in player aging and performance trajectories.

### Step 4.1: Fit XGBoost Regressor over Repeated Random Samples
Train `XGBRegressor` across 100 random splits to estimate metrics and predict targeted Corbin Carroll (`CB`) career outputs.

In [ ]:
for j in range(100):
    n_train_year = alltime1.sample(frac=0.8, replace=False, axis=0)
    n_test_year = alltime1.sample(frac=0.2, replace=False, axis=0)
    
    drop_cols_alltime = ['playerid', 'Name', 'Team', 'xwOBA', 'year', 'WAR', 'wRC', 'count', 'mlbamid', 
                         'Grade WAR', 'Grade AB', 'Grade H', 'Grade HR', 'Grade RBI', 'Grade SB', 'active']
    
    x_train_year = n_train_year.drop(drop_cols_alltime, axis=1, errors='ignore').select_dtypes(include=[np.number]).fillna(0)
    y_train_year = n_train_year['year'].fillna(0)
    x_test_year = n_test_year.drop(drop_cols_alltime, axis=1, errors='ignore').select_dtypes(include=[np.number]).fillna(0)
    y_test_year = n_test_year['year'].fillna(0)
    CB_test_year = CB.drop(drop_cols_alltime, axis=1, errors='ignore').select_dtypes(include=[np.number]).fillna(0)

    model_year = XGBRegressor(objective='reg:squarederror', alpha=1)
    model_year.fit(x_train_year, y_train_year)
    pred = model_year.predict(x_test_year)
    CB_pred = model_year.predict(CB_test_year)
    rmse = np.sqrt(MSE(y_test_year, pred))
    
print('Final Iteration RMSE:', round(rmse, 4))
print('Corbin Carroll Projected Year Point Estimate:', CB_pred)
print('R squared training set:', round(model_year.score(x_train_year, y_train_year) * 100, 2))
print('R squared test set:', round(model_year.score(x_test_year, y_test_year) * 100, 2))


# Section 5: Decision Tree Stratification & Multi-Step Recursive Forecasting

We construct a Decision Tree Regressor to cluster players into performance regimes based on `age` and previous season hits (`H-1`), then apply XGBoost within each regime to recursively forecast future hits `H` over a 12-year window.

### Step 5.1: Build Decision Tree Trajectory Categories for Hits (H)
Segment player performance into category bins using `DecisionTreeRegressor`.

In [ ]:
xx = data1[['age', 'H-1']].fillna(0)
yy = data1[['H']].fillna(0)

decision_tree = DecisionTreeRegressor(max_depth=3, min_samples_leaf=100)
decision_tree = decision_tree.fit(xx, yy)
r = export_text(decision_tree)
print("Decision Tree Rules:\n", r)

predictions = decision_tree.predict(xx)
pss = sorted(list(set(predictions)))

xxx = xx.to_numpy()
yyy = yy.to_numpy()
s = np.array([[p] for p in predictions])
aab = np.concatenate((xxx, yyy, s), axis=1)
aab = pd.DataFrame(aab, columns=['age', 'H-1', 'H', 'category'])

# Evaluate XGBoost on each category
for i in pss:
    aabb = aab.loc[aab['category'] == i]
    aabb_train = aabb.head(int(len(aabb) * 0.8))
    aabb_test = aabb.tail(int(len(aabb) * 0.2))
    x11_train = aabb_train[['age', 'H-1']]
    y11_train = aabb_train[['H']]
    x11_test = aabb_test[['age', 'H-1']]
    y11_test = aabb_test[['H']]
    
    model75 = XGBRegressor(objective='reg:squarederror', alpha=1)
    model75.fit(x11_train, y11_train)
    pred = model75.predict(x11_test)
    rmse = np.sqrt(MSE(y11_test, pred))
    print(f"Category {i:.2f} RMSE: {rmse:.4f}")


### Step 5.2: 12-Year Recursive Hit Projections
Iteratively forecast future hits (`H`) and update player age across 12 future seasons.

In [ ]:
future_h = future.copy()
if 'H-1' not in future_h.columns:
    future_h['H-1'] = future_h['WAR-1'] * 25 # proxy initialization if H-1 absent

for j in range(12):
    BB = decision_tree.predict(future_h[['age', 'H-1']])
    for i in pss:
        aabb = aab.loc[aab['category'] == i]
        aabb_train = aabb.head(int(len(aabb) * 0.8))
        aabb_test = aabb.tail(int(len(aabb) * 0.2))
        x11_train = aabb_train[['age', 'H-1']]
        y11_train = aabb_train[['H']]
        x11_test = aabb_test[['age', 'H-1']]
        y11_test = aabb_test[['H']]
        
        model75 = XGBRegressor(objective='reg:squarederror', alpha=1)
        model75.fit(x11_train, y11_train)
        
        if i == BB[0]:
            result = model75.predict(future_h[['age', 'H-1']])
            future_h['age'] += 1
            future_h['H-1'] = result
            print(f"Season +{j+1}: Age {int(future_h['age'].values[0])}, Predicted Hits (H): {result[0]:.2f}")


# Section 6: Conformal Prediction Framework (Uncertainty Quantification)

In this section, we apply **Inductive Conformal Prediction (ICP)** using the `nonconformist` library. Conformal prediction produces finite-sample, distribution-free 95% prediction intervals with proven coverage guarantees.

### Step 6.1: Visualization Utility for Conformal Bands
Define `plot_func` helper function to render scatter plots with prediction intervals and upper/lower bounds.

In [ ]:
split_color = 'tomato'
local_color = 'gray'
cqr_color = 'lightblue'

alphas = 0.05
quantiles = [5, 95]
max_show = 1000
save_figures = False

def plot_func(x, y, y_u=None, y_l=None, pred=None, shade_color="", method_name="", title="", filename=None, save_figures=False):
    x_ = x[:max_show]
    y_ = y[:max_show]
    if y_u is not None:
        y_u_ = y_u[:max_show]
    if y_l is not None:
        y_l_ = y_l[:max_show]
    if pred is not None:
        pred_ = pred[:max_show]

    fig = plt.figure()
    inds = np.argsort(np.squeeze(x_))
    plt.plot(x_[inds,:], y_[inds], 'k.', alpha=.2, markersize=10, fillstyle='none', label=u'Observations')
    
    if (y_u is not None) and (y_l is not None):
        plt.fill(np.concatenate([x_[inds], x_[inds][::-1]]),
                 np.concatenate([y_u_[inds], y_l_[inds][::-1]]),
                 alpha=.3, fc=shade_color, ec='None',
                 label = method_name + ' prediction interval')
    
    if pred is not None:
        if pred_.ndim == 2:
            plt.plot(x_[inds,:], pred_[inds,0], 'k', lw=2, alpha=0.9, label=u'Predicted low and high quantiles')
            plt.plot(x_[inds,:], pred_[inds,1], 'k', lw=2, alpha=0.9)
        else:
            plt.plot(x_[inds,:], pred_[inds], 'k--', lw=2, alpha=0.9, label=u'Predicted value')
    
    plt.ylim([-2.5, 40])
    plt.xlabel('$X$')
    plt.ylabel('$Y$')
    plt.legend(loc='upper right')
    plt.title(title)
    if save_figures and (filename is not None):
        plt.savefig(filename, bbox_inches='tight', dpi=300)
    
    plt.show()


### Step 6.2: Train, Calibration, and Test Split for ICP
Split `alltime1` into 60% proper training, 30% calibration set (for empirical error quantiles), and 10% test set.

In [ ]:
# Define base estimator
model_year = XGBRegressor(objective='reg:squarederror', alpha=1)
nc = RegressorNc(model_year, AbsErrorErrFunc())
icp = IcpRegressor(nc)

# Create 60/30/10 split
n_train_year = alltime1.sample(frac=0.6, replace=False, axis=0, random_state=1)
n_calibration_year = alltime1.sample(frac=0.3, replace=False, axis=0, random_state=1)
n_test_year = alltime1.sample(frac=0.1, replace=False, axis=0, random_state=1)

drop_cols_alltime = ['playerid', 'Name', 'Team', 'xwOBA', 'year', 'WAR', 'wRC', 'count', 'mlbamid', 
                     'Grade WAR', 'Grade AB', 'Grade H', 'Grade HR', 'Grade RBI', 'Grade SB', 'active']

x_train_year = n_train_year.drop(drop_cols_alltime, axis=1, errors='ignore').select_dtypes(include=[np.number]).fillna(0).to_numpy()
y_train_year = n_train_year['year'].fillna(0).to_numpy()

x_calibration_year = n_calibration_year.drop(drop_cols_alltime, axis=1, errors='ignore').select_dtypes(include=[np.number]).fillna(0).to_numpy()
y_calibration_year = n_calibration_year['year'].fillna(0).to_numpy()

x_test_year = n_test_year.drop(drop_cols_alltime, axis=1, errors='ignore').select_dtypes(include=[np.number]).fillna(0).to_numpy()
y_test_year = n_test_year['year'].fillna(0).to_numpy()

CB_test_year = CB.drop(drop_cols_alltime, axis=1, errors='ignore').select_dtypes(include=[np.number]).fillna(0).to_numpy()


### Step 6.3: Fit ICP Model & Evaluate Empirical Coverage
Fit underlying regressor on training set, calibrate nonconformity scores on calibration set, and predict 95% confidence intervals on test set.

In [ ]:
# Fit on proper training data
icp.fit(x_train_year, y_train_year)

# Calibrate error residuals
icp.calibrate(x_calibration_year, y_calibration_year)

# Predict intervals at alpha = 0.05 (95% confidence)
predictions = icp.predict(x_test_year, significance=alphas)
y_lower = predictions[:, 0]
y_upper = predictions[:, 1]

# Calculate coverage percentage
in_the_range = np.sum((y_test_year >= y_lower) & (y_test_year <= y_upper))
coverage = in_the_range / len(y_test_year) * 100
avg_length = np.mean(y_upper - y_lower)

print(f"Empirical Coverage: {coverage:.2f}% (Target: {100*(1-alphas):.1f}%)")
print(f"Average Interval Length: {avg_length:.4f}")


# Section 7: Advanced Conformal Prediction & Multi-Step Projections (AB & WAR)

In this final section, we extend conformal prediction to multi-step recursive projections for At Bats (`AB`) and Wins Above Replacement (`WAR`) across a player's career trajectory, and evaluate Normalized ICP with heteroscedastic error bounds.

### Step 7.1: Stratified Conformal Forecasting for At-Bats (AB)
Forecast At Bats (`AB`) over 12 future seasons incorporating conformal error bounds.

In [ ]:
future_ab = future.copy()
if 'H-1' not in future_ab.columns:
    future_ab['H-1'] = future_ab['WAR-1'] * 25

for j in range(12):
    BB = decision_tree.predict(future_ab[['age', 'H-1']])
    for i in pss:
        aabb = aab.loc[aab['category'] == i]
        aabb_train = aabb[:int(len(aabb)*0.6)]
        aabb_calibration = aabb[int(len(aabb)*0.6):int(len(aabb)*0.8)]
        aabb_test = aabb[int(len(aabb)*0.9):]
        
        x11_train = aabb_train[['age', 'H-1']].to_numpy()
        y11_train = aabb_train['H'].to_numpy()
        
        x11_calibration = aabb_calibration[['age', 'H-1']].to_numpy()
        y11_calibration = aabb_calibration['H'].to_numpy()
        
        x11_test = aabb_test[['age', 'H-1']].to_numpy()
        y11_test = aabb_test['H'].to_numpy()
        
        if len(x11_train) > 0 and len(x11_calibration) > 0 and len(x11_test) > 0:
            icp.fit(x11_train, y11_train)
            icp.calibrate(x11_calibration, y11_calibration)
            predictions = icp.predict(x11_test, significance=alphas)
            y_lower = predictions[:, 0]
            y_upper = predictions[:, 1]
            length_split_rf = y_upper - y_lower
            
            if i == BB[0]:
                result = model75.predict(future_ab[['age', 'H-1']])
                future_ab['age'] += 1
                future_ab['H-1'] = result - (np.mean(length_split_rf)) / 2
                print(f"Season +{j+1}: Age {int(future_ab['age'].values[0])}, Proj Stat with Conformal Bounds: {result[0]:.2f}")


### Step 7.2: Normalized ICP with Heteroscedastic Error Scaling
Build a `RegressorNormalizer` with two XGBoost models (conditional mean model and MAD error model) to adjust interval width dynamically according to local difficulty.

In [ ]:
model_year = XGBRegressor(objective='reg:squarederror', alpha=1)
med_year = XGBRegressor(objective='reg:squarederror', alpha=1)

normalizer = RegressorNormalizer(model_year, med_year, AbsErrorErrFunc())
nc = RegressorNc(model_year, AbsErrorErrFunc(), normalizer)
icp = IcpRegressor(nc)

coverages = []
lengths = []

drop_cols_alltime = ['playerid', 'Name', 'Team', 'xwOBA', 'year', 'WAR', 'wRC', 'count', 'mlbamid', 
                     'Grade WAR', 'Grade AB', 'Grade H', 'Grade HR', 'Grade RBI', 'Grade SB', 'active']

for j in range(10): # Run 10 trials for demonstration
    n_train_year = alltime1.sample(frac=0.6, replace=False, axis=0)
    n_calibration_year = alltime1.sample(frac=0.3, replace=False, axis=0)
    n_test_year = alltime1.sample(frac=0.1, replace=False, axis=0)
    
    x_tr = n_train_year.drop(drop_cols_alltime, axis=1, errors='ignore').select_dtypes(include=[np.number]).fillna(0).to_numpy()
    y_tr = n_train_year['year'].fillna(0).to_numpy()
    x_cal = n_calibration_year.drop(drop_cols_alltime, axis=1, errors='ignore').select_dtypes(include=[np.number]).fillna(0).to_numpy()
    y_cal = n_calibration_year['year'].fillna(0).to_numpy()
    x_te = n_test_year.drop(drop_cols_alltime, axis=1, errors='ignore').select_dtypes(include=[np.number]).fillna(0).to_numpy()
    y_te = n_test_year['year'].fillna(0).to_numpy()
    
    icp.fit(x_tr, y_tr)
    icp.calibrate(x_cal, y_cal)
    
    preds = icp.predict(x_te, significance=alphas)
    y_low, y_high = preds[:, 0], preds[:, 1]
    
    cov = np.sum((y_te >= y_low) & (y_te <= y_high)) / len(y_te) * 100
    avg_len = np.mean(y_high - y_low)
    coverages.append(cov)
    lengths.append(avg_len)

print(f"Normalized ICP Mean Coverage (10 runs): {np.mean(coverages):.2f}%")
print(f"Normalized ICP Mean Interval Length: {np.mean(lengths):.4f}")


### Step 7.3: 14-Year Stratified Conformal Projections for Wins Above Replacement (WAR)
Construct Decision Tree regimes on `['age', 'WAR-1']` to project 14 years of future player WAR metrics with conformal uncertainty bounds.

In [ ]:
xx_war = data1[['age', 'WAR-1']].fillna(0)
yy_war = data1[['WAR']].fillna(0)

decision_tree_war = DecisionTreeRegressor(max_depth=3, min_samples_leaf=100)
decision_tree_war = decision_tree_war.fit(xx_war, yy_war)

preds_war = decision_tree_war.predict(xx_war)
pss_war = sorted(list(set(preds_war)))

xxx_w = xx_war.to_numpy()
yyy_w = yy_war.to_numpy()
s_w = np.array([[p] for p in preds_war])
aab_war = np.concatenate((xxx_w, yyy_w, s_w), axis=1)
aab_war = pd.DataFrame(aab_war, columns=['age', 'WAR-1', 'WAR', 'category'])

future_war = future.copy()

print("Beginning 14-Year Conformal WAR Trajectory Forecasting:")
for j in range(14):
    f_input = future_war[['age', 'WAR-1']]
    BB = decision_tree_war.predict(f_input)
    for i in pss_war:
        aabb = aab_war.loc[aab_war['category'] == i]
        if len(aabb) > 10:
            aabb_train = aabb[:int(len(aabb)*0.8)]
            aabb_calibration = aabb[int(len(aabb)*0.8):int(len(aabb)*0.9)]
            aabb_test = aabb[int(len(aabb)*0.9):]
            
            x11_train = aabb_train[['age', 'WAR-1']].to_numpy()
            y11_train = aabb_train['WAR'].to_numpy()
            x11_calibration = aabb_calibration[['age', 'WAR-1']].to_numpy()
            y11_calibration = aabb_calibration['WAR'].to_numpy()
            x11_test = aabb_test[['age', 'WAR-1']].to_numpy()
            y11_test = aabb_test['WAR'].to_numpy()
            
            fu = f_input.to_numpy()
            test_future = np.vstack((x11_test, fu))
            
            icp.fit(x11_train, y11_train)
            icp.calibrate(x11_calibration, y11_calibration)
            
            pred = model_year.predict(test_future)
            y_future = pred[-1]
            
            preds_conf = icp.predict(test_future, significance=alphas)
            y_low_f, y_up_f = preds_conf[-1, 0], preds_conf[-1, 1]
            
            if i == BB[0]:
                curr_age = future_war['age'].values[0]
                print(f"Season +{j+1} (Age {int(curr_age)}): Proj WAR = {y_future:.2f} [95% CI: {y_low_f:.2f}, {y_up_f:.2f}]")
                future_war['age'] += 1
                future_war['WAR-1'] = y_future
                if 'year' in future_war.columns:
                    future_war['year'] += 1
